# [Introduction to Data Science](http://datascience-intro.github.io/1MS041-2026/)    
## 1MS041, 2026 
&copy;2026 Raazesh Sainudiin, Benny Avelin. [Attribution 4.0 International     (CC BY 4.0)](https://creativecommons.org/licenses/by/4.0/)

# Lecture 11: building linear and kernel classifiers

This notebook accompanies Sections 8.1--8.3. Labels are in
$\{-1,1\}$. An affine score is $f_{w,b}(x)=w\cdot x+b$, and the
classifier returns $1$ when the score is positive and $-1$
otherwise.


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from sklearn.datasets import load_digits
from sklearn.metrics import accuracy_score, precision_score, recall_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC

rng = np.random.default_rng(2026)
np.set_printoptions(precision=4, suppress=True)


## Strict separation and the intercept term

A finite labelled data set is linearly separable if some $w,b$
satisfy $y_i(w\cdot x_i+b)>0$ for every $i$. Set
$\widetilde x_i=(x_i,1)$ and $\widetilde w=(w,b)$. The last
coordinate has a plus sign, so
$\widetilde w\cdot\widetilde x_i=w\cdot x_i+b$.


In [ ]:
X_sep = np.array([
    [-2.0, -1.0], [-1.8, 0.2], [-1.3, 1.0],
    [ 1.2, -1.1], [ 1.7, 0.1], [ 2.2, 1.1],
])
y_sep = np.array([-1, -1, -1, 1, 1, 1])
X_aug = np.column_stack([X_sep, np.ones(len(X_sep))])
separator = np.array([1.0, 0.0, 0.0])
signed_margins = y_sep * (X_aug @ separator)
print("signed margins:", signed_margins)
print("strictly separated:", np.all(signed_margins > 0))


## How the perceptron learns

Starting from zero, update when a signed margin is nonpositive:
$$
y_i(\widetilde w_j\cdot\widetilde x_i)\leq0
\quad\Longrightarrow\quad
\widetilde w_{j+1}=\widetilde w_j+y_i\widetilde x_i.
$$
A complete pass with no update certifies strict separation.


In [ ]:
def perceptron(X, y, max_epochs=1_000):
    X_aug = np.column_stack([np.asarray(X, float), np.ones(len(X))])
    y = np.asarray(y, int)
    w, updates = np.zeros(X_aug.shape[1]), 0
    for epoch in range(1, max_epochs + 1):
        changed = False
        for xi, yi in zip(X_aug, y):
            if yi * (w @ xi) <= 0:
                w += yi * xi
                updates += 1
                changed = True
        if not changed:
            return w, updates, epoch
    raise RuntimeError("no separator found within max_epochs")

w_hat, updates, epochs = perceptron(X_sep, y_sep)
fitted_margins = y_sep * (X_aug @ w_hat)
print("augmented (w,b):", w_hat)
print("updates/passes:", updates, epochs)
print("minimum fitted margin:", fitted_margins.min())

plt.scatter(X_sep[:, 0], X_sep[:, 1], c=y_sep, cmap="coolwarm", s=70)
x_grid = np.linspace(-2.6, 2.6, 200)
if abs(w_hat[1]) > 1e-12:
    plt.plot(x_grid, -(w_hat[0] * x_grid + w_hat[2]) / w_hat[1], "k-")
else:
    plt.axvline(-w_hat[2] / w_hat[0], color="black")
plt.xlabel("$x_1$")
plt.ylabel("$x_2$")
plt.show()


If $w^*$ has normalized margins
$y_i(w^*\cdot\widetilde x_i)\geq1$, and
$r=\max_i|\widetilde x_i|$, the perceptron makes at most
$r^2|w^*|^2$ updates. The level $1$ is a normalization: a strict
separator for a finite data set can be rescaled to achieve it.


In [ ]:
w_star = separator / signed_margins.min()
r = np.linalg.norm(X_aug, axis=1).max()
bound = r**2 * np.linalg.norm(w_star)**2
print("normalized margins:", y_sep * (X_aug @ w_star))
print(f"observed updates={updates}; theorem bound={bound:.2f}")


## Using kernels to compare observations

A function $k$ is a kernel when every finite Gram matrix
$K_{ij}=k(x_i,x_j)$ is symmetric positive semidefinite. If
$k(x,z)=\phi(x)\cdot\phi(z)$, then
$a^TKa=|\sum_i a_i\phi(x_i)|^2\geq0$.

For $x\in\mathbb R^2$, the kernel $k(x,z)=(1+x\cdot z)^2$
corresponds to
$$
\phi(x)=(1,\sqrt2x_1,\sqrt2x_2,x_1^2,
         \sqrt2x_1x_2,x_2^2).
$$


In [ ]:
def polynomial_kernel(X, Z):
    return (1 + np.asarray(X) @ np.asarray(Z).T) ** 2

def phi_degree_two(X):
    X = np.asarray(X)
    x1, x2 = X[:, 0], X[:, 1]
    return np.column_stack([
        np.ones(len(X)), np.sqrt(2) * x1, np.sqrt(2) * x2,
        x1**2, np.sqrt(2) * x1 * x2, x2**2,
    ])

X_kernel = rng.normal(size=(8, 2))
K = polynomial_kernel(X_kernel, X_kernel)
Phi = phi_degree_two(X_kernel)
print("feature identity error:", np.max(np.abs(K - Phi @ Phi.T)))
print("smallest Gram eigenvalue (floating-point roundoff):",
      np.linalg.eigvalsh(K).min())


## Allowing mistakes with hinge loss and a soft margin

For score $f(x)$, the hinge loss is
$$
\ell_{\rm hinge}(y,f(x))=\max\{0,1-yf(x)\}.
$$
The maximum is essential. Correct predictions with signed margin at
least $1$ have zero loss; points inside the margin or on the wrong
side have positive loss. A soft-margin SVM balances a quadratic
penalty on $w$ with a sum of hinge losses. Kernel SVMs express the
score through kernel evaluations.


In [ ]:
def hinge_loss(y, score):
    return np.maximum(0, 1 - np.asarray(y) * np.asarray(score))

demo_margins = np.array([1.5, 1.0, 0.4, -0.5])
print("signed margins:", demo_margins)
print("hinge losses:", np.maximum(0, 1 - demo_margins))


## One train/test split on the digits data

We predict whether a digit is at least $5$. Scaling is fitted on the
training data inside each pipeline, and both SVMs use the same split.
The method named score on an SVC reports accuracy, not precision.


In [ ]:
digits = load_digits()
X_digits = digits.data
y_digits = (digits.target >= 5).astype(int)
X_train, X_test, y_train, y_test = train_test_split(
    X_digits, y_digits, test_size=0.25, stratify=y_digits,
    random_state=2026
)
models = {
    "linear": make_pipeline(
        StandardScaler(), SVC(kernel="linear", C=1)
    ),
    "degree-2 polynomial": make_pipeline(
        StandardScaler(),
        SVC(kernel="poly", degree=2, coef0=1, C=1),
    ),
}
for name, model in models.items():
    model.fit(X_train, y_train)
    prediction = model.predict(X_test)
    print(
        f"{name:19s} accuracy={accuracy_score(y_test, prediction):.3f} "
        f"precision={precision_score(y_test, prediction):.3f} "
        f"recall={recall_score(y_test, prediction):.3f}"
    )


## Try a nonlinear boundary

The next cell generates an inner disc (label $1$) and an outer
annulus (label $-1$), then makes one fixed split.

1. Explain why no affine line separates the two complete regions.
2. Run the supplied linear and degree-two polynomial SVMs. Which
   feature in $\phi$ separates radius classes?
3. Change the seed or sample size, but keep the same split for both
   models and compare held-out rather than training accuracy.

The cell is complete and reproducible; edit the model dictionary for
further comparisons.


In [ ]:
n_each = 300
theta_inner = rng.uniform(0, 2 * np.pi, n_each)
theta_outer = rng.uniform(0, 2 * np.pi, n_each)
radius_inner = np.sqrt(rng.uniform(0, 1, n_each))
radius_outer = np.sqrt(rng.uniform(3**2, 4**2, n_each))
inner = radius_inner[:, None] * np.column_stack([
    np.cos(theta_inner), np.sin(theta_inner)
])
outer = radius_outer[:, None] * np.column_stack([
    np.cos(theta_outer), np.sin(theta_outer)
])
X_ring = np.vstack([inner, outer])
y_ring = np.r_[np.ones(n_each, int), -np.ones(n_each, int)]
Xr_train, Xr_test, yr_train, yr_test = train_test_split(
    X_ring, y_ring, test_size=0.30, stratify=y_ring, random_state=2026
)
student_models = {
    "linear": SVC(kernel="linear", C=10),
    "degree-2 polynomial": SVC(
        kernel="poly", degree=2, gamma=1, coef0=1, C=10
    ),
}
for name, model in student_models.items():
    model.fit(Xr_train, yr_train)
    print(name, "held-out accuracy =", model.score(Xr_test, yr_test))

plt.scatter(Xr_test[:, 0], Xr_test[:, 1], c=yr_test,
            cmap="coolwarm", s=12)
plt.gca().set_aspect("equal")
plt.show()
